In [1]:
# Initialize Otter
import otter
grader = otter.Notebook("week10.ipynb")

# HW10: Architecture of an Inventory Robot
#### CS-31: Foundations for Artificial Intelligence
*Point breakdown: 90 manually graded · 10 collaboration statement = 100*


*You are allowed to work in groups for the lab. You must write anything that will be turned in -- all code and all written solutions -- on your own. You must list all people and resources that you interacted with in the Collaboration Statement below. See [course policies](https://www.cs.tufts.edu/comp/31/policy.html) for more information.*

--

Remember back to our first lecture, where we discussed how AI is a collection of technologies. I gave a motivating example of a [store inventory robot](https://www.simberobotics.com/), which roams around a store and automatically checks and updates inventory. For this week's lab, I want you to design and describe a hypothetical control architecture for a store inventory robot using a hierarchical control style.

Your overarching goal is to design the robot's control architecture such that it can "autonomously navigate a store to check and update inventory." **It is up to *you* to specify this broad goal and break it down into subcomponents, the tasks it needs to complete, and the actions it needs to take.** I would recommend sketching this out and doing some free-form ideation before actually writing your answers in this notebook!

Some guidelines:
- Feel free to make any assumptions you wish about what data or knowledge the agent has beforehand, as well as any sensors or actuators it might have—but you must state these assumptions.
- **Your architecture should have a minimum of three control layers, to a maximum of five.**
- You should also keep in mind that the layers of your architecture that achieve this goal don't all have to be *physically* on the robot itself (i.e., what operations and functionality can lie on a computer or cloud, and what capabilities should be on the actual robot itself?).
- Feel free to include attachments of images or diagrams if you want to draw this out - submit your assignment as a *.zip.
- Some links for research and inspiration:
    - Textbook: For clear definitions of percepts, commands, belief states, layers, etc. As well as examples of breaking down tasks/layers.
        - Textbook [2.1 Agents and Environments](https://artint.info/3e/html/ArtInt3e.Ch2.S1.html)
        - Textbook [2.2 Hierarchical Control](https://artint.info/3e/html/ArtInt3e.Ch2.S2.html)
        - Textbook [2.3 Designing Agents](https://artint.info/3e/html/ArtInt3e.Ch2.S3.html)
    - [Tally Page](https://www.simberobotics.com/store-intelligence/tally) gives a short (albeit marketing-coded) overview of the features of the inventory robot (such as its sensors and how it moves).
    - [Simbe Platform Demo](https://www.simberobotics.com/resources/insights/simbe-platform-demo) gives an overview of what a store manager might see using the information collected by the robot, and how they could control it.
    - Feel free to bring in any other research or sources.


### **Table of Contents**

1) Overview
2) Detailed Layer Descriptions
3) Behavior Trace
4) Reflection
5) Collaboration Statement


**AI / Source Disclosure**

ChatGPT drafted the architecture and written responses in Sections 1–4 from the assignment prompt. These are working answers for revision in my own words.

## **1. Overview**


<!-- BEGIN QUESTION -->

#### **Q1: Overview (10 pts)**

Give an overall summary of the architecture. What does each component do at a high level? What type of information does it process? Include the assumptions you made about the robot's data, knowledge, sensors, and actuators. (It might make the most sense to answer this question last, after your initial design.)


The robot audits visible shelf stock. A store server turns a manager’s request into a list of shelf bays, an onboard task controller plans how to visit and scan each bay, and an onboard hardware controller moves the robot and collects observations. The server records a timestamped visible count for each product and flags uncertain results for staff. A shelf count measures what a shopper can see. Backroom stock and items hidden behind the front row require separate information.

I assume the store supplies a floor map with aisle intersections and scan positions, a planogram mapping product IDs to shelf slots, and labeled reference images. The inventory system has a separate field for visible counts. The robot has wheel motors and encoders, a depth sensor for nearby obstacles, a bumper, and a shelf-facing camera on a motorized mast. Each bay has a readable location marker. The robot can stop, adjust the camera height, and scan overlapping views. It can cache observations if its connection to the server drops.

<!-- END QUESTION -->

## **2. Detailed Layer Descriptions**


<!-- BEGIN QUESTION -->

#### **Q2: Detailed Layer Descriptions (60 points)**

Clearly describe each layer and its functionality. For each layer, clearly describe its function and how it is distinct from its neighboring layers. For example, in our simple delivery robot, the top layer only managed the location queue, the middle layer managed navigation, and the lower layer turned those navigation commands into physical actions.

- Your description of each layer should include:
    - Roughly, what is the planning horizon of this layer? Give a time bound on the order of seconds, minutes, etc that this layer operates at.
    - A paragraph describing the broad overall function of the layer and its role in the agent. What does it do? What does it *not* do?
    - A clear description of what functions this layer may have, and what memories or variables it may keep track of (i.e., what does the layer's belief state look like?).
        - For discussing the functions of each layer, you **must** reference at least one algorithmic technique or method we discussed in this class that could be used to implement that function.
            - For example, in the delivery robot, the top layer could frame this as a pathfinding problem and use our pathfinding algorithms to solve it, then pass that path down for the lower layers to execute.
        - For describing memory/belief states, you should give a rationale for why the agent is storing these variables, and also what information that layer is *not* storing or filtering out.
    - A detailed description of the input and output commands/percepts for each layer:
        - Commands:
            - Input commands: What information and commands does the layer get from above?
            - Output commands: What information and commands does this layer give to the layer below it?
        - Percepts:
            - Input percepts: What information does this layer get from the layer below it?
            - Output percepts: What information does this layer give to the layer above it?
        - Prior belief state and next belief states
        - When discussing percepts, commands and belief states, **you should clearly detail how each layer is transforming, processing, and filtering information for the layers above and below it.** What information do you pass on? What information do you *not* give?
            - For example, in our delivery robot, the top layer that manages the location queue does not need to know about the exact position of the robot, its current velocity, or the amount of power needed to drive the wheels. Correspondingly, the bottom layer that drives the motors and takes in raw sensor input does not need to know about the robot's overall goal—only how to control the hardware at the current instant.


Layer 1 — Audit planning and inventory records (store server, minutes to hours). The manager supplies a product category, store area, and priority. The server looks up the relevant slots and groups them by bay. It uses a greedy search over unfinished bays, choosing the nearest reachable bay of the highest pending priority. Its downward command identifies the audit, bay, and requested slots. The onboard controller chooses the driving route. Grouping slots lets one stop serve several products.

Before a request, the server believes each slot has its last recorded visible count and timestamp. It keeps these alongside the pending bays and audit ID so it can tell an old observation from a fresh one. A report from below contains the bay, slot, product ID, count, observation time, and confidence or failure reason. A verified report changes the slot from pending to counted and updates its visible count. An unreadable slot goes into a review queue and retains its previous count and timestamp. The server sends the manager a summary of changes and unresolved slots. It discards motion details such as wheel speed and steering corrections.

Layer 2 — Navigation and shelf inspection (onboard, seconds to minutes). Given a bay and slot list, this layer plans a route with A* on a map graph. Edge costs are travel distances, and straight-line distance to the bay is a lower bound on the remaining distance. It sends one waypoint at a time to Layer 3. At the bay it checks the location marker, then requests captures at the heights and positions needed to cover the assigned slots.

Its prior belief contains an estimated pose, a route, and a list of unseen slots. It keeps temporary blocked edges for replanning and scan coverage to avoid counting an item twice. From below it receives pose estimates, waypoint status, blockage reports, and tagged images. A persistent blockage changes the current graph and triggers a new route search. A capture changes a slot’s state from unseen to observed. The controller matches visible packages to reference images and projects detections from overlapping views onto shelf coordinates so each distinct facing contributes once. A clear view of a fully covered empty slot supports a zero. An obstructed view triggers a second scan from another position and then a review flag if still unclear. Layer 1 receives one result per slot, with confidence and time. Images and intermediate route choices stay onboard.

Layer 3 — Motion, sensing, and immediate stopping (onboard, milliseconds to seconds). This layer receives a waypoint, a stop command, or a request to capture an image at a given mast height. A condition–action controller checks the bumper and depth readings each cycle. A nearby obstacle triggers a stop before any waypoint-following action. With a clear path, the controller adjusts wheel speeds from heading and distance error. For a capture it stops the base, positions the mast, and triggers the camera.

Its prior state includes the last pose estimate, encoder readings, active target, and motion state. New encoder readings update the pose, and a new depth reading can switch the state from advancing to stopped. It sends wheel and mast commands directly to actuators and receives raw encoder, depth, bumper, and camera data from the hardware. It reports estimated pose and events such as waypoint reached or path blocked, plus the requested image tagged with pose and mast height. Raw encoder ticks are consumed locally. This layer needs the current target but has no use for a product category or the store’s inventory records.

<!-- END QUESTION -->

## **3. Behavior Trace**


<!-- BEGIN QUESTION -->

#### **Q3: Behavior Trace (10 pts)**

Consider the following scenario: A store manager gets the feeling that they should re-check the stock for milk and dairy products — so they send a command for the inventory robot to perform a recount.

Write a paragraph tracing this command down and up through the layers, detailing what the robot actually does at each layer to accomplish this task. You should be clear about how the commands/percepts change at each boundary between layers, and what gets passed up/down and what is filtered out. Your answer should be detailed enough that, by the end of it, this hypothetical store manager has a decent idea of how the robot works and how information flows between each layer.


Suppose the manager asks for a dairy recount. Layer 1 looks up the dairy slots and sends Layer 2 an audit ID and a first bay containing milk and yogurt. Layer 2 uses A* to plan a route and sends the next waypoint to Layer 3. When a cart enters the aisle, Layer 3 stops the wheels and reports a blockage and its location. Layer 2 can plan around the cart without involving the manager. At the bay, Layer 2 verifies the location marker and requests stationary scans at the milk and yogurt shelf heights. Layer 3 returns images tagged with position and height. Layer 2 combines overlapping views and reports six visible milk cartons. A sign hides part of the yogurt slot even after a second scan, so it reports incomplete coverage for yogurt. Layer 1 records six as the fresh visible milk count, keeps the earlier yogurt count with its original timestamp, and queues yogurt for a staff check. After the remaining bays, the manager sees which counts changed and which still need attention. Wheel commands, camera frames, and the temporary detour stay onboard.

<!-- END QUESTION -->

## **4. Reflection**


<!-- BEGIN QUESTION -->

#### **Q4: Reflection (10 pts)**

Reflect. Answer the following questions in a paragraph or two:
- Why did you break up the architecture in this way?
- Could you see a different architecture that would also work (e.g., merging two layers into one or splitting one layer into multiple)?
- What are some potential issues that you might see with this architecture, both technically and in terms of broader implications? Think about unintended consequences (as we discussed in the early weeks) and possible stakeholder harms (as we discussed in the ethics week). How would you address these?


I chose three layers because the timing and information needed for each decision differ. The wheels need to stop as soon as a cart appears, while the manager can wait several minutes for a recount. Keeping the audit queue on the server also lets staff inspect outstanding work while the robot handles interruptions locally. A four-layer design could separate shelf recognition from navigation. That would let a team change the counting algorithm independently, at the cost of having to synchronize the camera pose and the shelf location across another interface.

An incorrect zero could send a worker to refill a full shelf or lead to an unnecessary order. The robot therefore tracks coverage, records uncertainty, and asks for a manual check when a second view still leaves a slot obscured. A stopped robot can also block a shopper using a wheelchair, so the route map should include accessible clearances and designated waiting areas. Finally, shelf cameras may capture customers or staff. The system can aim the camera toward the shelf, process images onboard, discard frames after counting, and send only slot-level results to the server. Staff should be able to correct a count and see why it was flagged.

<!-- END QUESTION -->

## **5. Collaboration Statement**

You are required to complete the following collaboration statement.


<!-- BEGIN QUESTION -->

- If you completed the lab in a group - list all your group members here, including yourself. **If you worked individually, you must state *"I worked by myself."***
    - I worked by myself.
- You must list the names of any others you got help from when completing this assignment (including TAs), and in what way you worked with them (discussed ideas, debugged math). **If you did not collaborate at all, you must state *"I did not collaborate with anyone else."***
    - I did not collaborate with anyone else.
- If you found any resources online helpful, you must link them and describe how you found them helpful (or not helpful). **If you did not use any resources online, you must state *"I did not use any resources online."***
    - I did not use any resources online.
- If you used any AI assistive technologies, you must describe how you used them and how you found them helpful (or not helpful). **If you did not use any AI assistive technologies, you must state *"I did not use any AI assistive technologies."***
    - I used LLM to help me connect each layer’s commands to the observations returned by the layer below.
- If you have any questions or comments about the material, or there was a topic or question you are confused about -- please make a note of it here! **If you do not have any questions or comments, you must state *"I do not have any questions or comments about the assignment."***
    - I do not have any questions or comments about the assignment.
- Estimate how long this assignment took you in hours. **You must give an estimate**.
    - This assignment took me about 3 hours to complete.



<!-- END QUESTION -->



## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

Run every cell from top to bottom and SAVE before exporting. Then submit
the generated .zip to the HW10 autograder assignment on Gradescope.


In [2]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False, run_tests=True)

Running your submission against local test cases...


Your submission received the following results when run against available test cases:




/Users/arthur/Downloads/Git/cs31_env/.venv/lib/python3.13/site-packages/otter/check/utils.py:115: RuntimeWarning: [IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.

  warnings.warn(results.stderr.decode("utf-8"), RuntimeWarning)
